# Public matched-dataset augmentation
Evaluate whether public prompt blocks improve unseen-prompt generalization. The external dataset is used only as public training data; validation rows remain competition train rows held out by prompt_id.


In [ ]:
from pathlib import Path
import re, json
import numpy as np, pandas as pd
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.svm import LinearSVC
from sklearn.metrics import balanced_accuracy_score
DATA=Path('/kaggle/input/competitions/name-that-llm')
EXT_ROOT=Path('/kaggle/input')
OUT=Path('/kaggle/working/public_augmentation_results.json')
train=pd.read_csv(DATA/'train.csv')
matches=list(EXT_ROOT.rglob('passages_clean.csv'))
assert matches, f'Could not find passages_clean.csv under {EXT_ROOT}'
ext=pd.read_csv(matches[0]); print('external file',matches[0])
labels=sorted(train.model.unique()); assert set(ext.model).issubset(set(labels))
# Keep only complete 18-model prompt blocks to avoid skew from incomplete external blocks.
counts=ext.groupby(ext['id'].astype(str).str.split('__').str[0])['model'].nunique()
valid_prompts=set(counts[counts==18].index)
ext=ext[ext['id'].astype(str).str.split('__').str[0].isin(valid_prompts)].copy()
print('competition',len(train),'external',len(ext),'complete prompts',len(valid_prompts))
PUNCT=tuple(".,;:!?()[]{}\"'/ -".replace(" ",""))
def feats(items):
 rows=[]
 for s in items:
  s=str(s); den=max(len(s),1); words=re.findall(r"\b[\w']+\b",s)
  rows.append([len(s),len(words),len(re.findall(r'[.!?]+',s)),sum(s.count(c) for c in PUNCT),sum(c.isdigit() for c in s),sum(c.isupper() for c in s),s.count('?'),s.count('!'),s.count(':'),s.count(';'),s.count('\n')]+[s.count(c)/den for c in PUNCT])
 return np.asarray(rows,dtype=np.float32)
splitter=StratifiedGroupKFold(n_splits=3,shuffle=True,random_state=42)
splits=list(splitter.split(train,train.model,train.prompt_id))
results=[]
for fold,(tr,va) in enumerate(splits,1):
 base=train.iloc[tr]; hold=train.iloc[va]
 for name,use_ext,w_comp in [('competition_only',False,1.0),('public_augmented',True,2.0)]:
  learn=pd.concat([base,ext[['text','model']]],ignore_index=True) if use_ext else base[['text','model']].copy()
  texts=learn.text.fillna('').astype(str).to_numpy(); valtext=hold.text.fillna('').astype(str).to_numpy()
  word=TfidfVectorizer(analyzer='word',ngram_range=(1,3),min_df=2,sublinear_tf=True,max_features=50000)
  char=TfidfVectorizer(analyzer='char',ngram_range=(1,5),min_df=2,sublinear_tf=True,max_features=50000)
  Xw=word.fit_transform(texts); Xv_w=word.transform(valtext)
  Xc=char.fit_transform(texts); Xv_c=char.transform(valtext)
  sc=StandardScaler(); Xs=csr_matrix(sc.fit_transform(feats(texts))); Xv_s=csr_matrix(sc.transform(feats(valtext)))
  X=hstack([Xw,Xc,Xs],format='csr'); Xv=hstack([Xv_w,Xv_c,Xv_s],format='csr')
  weights=np.ones(len(learn));
  if use_ext: weights[:len(base)]=w_comp
  clf=LinearSVC(C=1.5,class_weight='balanced',max_iter=2500).fit(X,learn.model.to_numpy(),sample_weight=weights)
  pred=clf.predict(Xv); score=balanced_accuracy_score(hold.model,pred)
  results.append({'fold':fold,'method':name,'score':float(score),'rows':len(learn)})
  print(f'fold={fold} method={name} score={score:.5f}',flush=True)
summary={'results':results}
for method in sorted(set(r['method'] for r in results)):
 summary[method+'_mean']=float(np.mean([r['score'] for r in results if r['method']==method]))
OUT.write_text(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))
